# Empirical examples of facility location problems

*Authors:* [Germano Barcelos](https://github.com/gegen07), [James Gaboardi](https://github.com/jGaboardi), [Levi J. Wolf](https://github.com/ljwolf), [Qunshan Zhao](https://github.com/qszhao)

This tutorial aims to show a facility location application. To achieve this goal the tuorial will solve mulitple real world facility location problems using a dataset describing an area of census tract 205, San Francisco. The general scenarios can be stated as: store sites should supply the demand in this census tract considering the distance between the two sets of sites: demand points and candidate supply sites. Four fundamental facility location models are utilized to highlight varying outcomes dependent on objectives: LSCP (Location Set Covering Problem), MCLP (Maximal Coverage Location Problem), P-Median and P-Center. For further information on these models, it's recommended to see the notebooks that explain more deeply about each one: [LSCP](./lscp.ipynb), [MCLP](./mclp.ipynb), [P-Median](./p-median.ipynb), [P-Center](./p-center.ipynb).

Also, this tutorial demonstrates the use of different solvers that [PULP](https://github.com/coin-or/pulp) supports.

In [ ]:
%config InlineBackend.figure_format = "retina"
%load_ext watermark
%watermark

In [ ]:
import time
import warnings
from pathlib import Path

import geopandas
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import matplotlib_scalebar
import numpy
import pandas
import pulp
import shapely
from matplotlib.patches import Patch
from matplotlib_scalebar.scalebar import ScaleBar
from shapely.geometry import Point

import spopt

%watermark -w
%watermark -iv

-------------------
We use 4 data files as input:
- `network_distance` is the distance between facility candidate sites calculated by ArcGIS Network Analyst Extension
- `demand_points` represents the demand points with some important features for the facility location problem like population
- `facility_points` represents the stores that are candidate facility sites
- `tract` is the polygon of census tract 205.

All datasets are online on this [repository](https://github.com/huanfachen/Open_source_location_cover_models/tree/master/Data/San_Francisco_store).

In [ ]:
DIRPATH = Path(spopt.__file__).parent.joinpath("tests/test_locate/data/")

`network_distance` dataframe

In [ ]:
network_distance = pandas.read_csv(
    DIRPATH.joinpath("SF_network_distance_candidateStore_16_censusTract_205_new.csv")
)
network_distance

`demand_points` dataframe

In [ ]:
demand_points = pandas.read_csv(
    DIRPATH.joinpath("SF_demand_205_centroid_uniform_weight.csv"), index_col=0
)
demand_points = demand_points.reset_index(drop=True)
demand_points

`facility_points` dataframe

In [ ]:
facility_points = pandas.read_csv(DIRPATH.joinpath("SF_store_site_16_longlat.csv"), index_col=0)
facility_points = facility_points.reset_index(drop=True)
facility_points

`study_area` dataframe

In [ ]:
study_area = geopandas.read_file(DIRPATH.joinpath("ServiceAreas_4.shp")).dissolve()
study_area

Plot `study_area`

In [ ]:
base = study_area.plot()
base.axis("off");

To start modeling the problem assuming the arguments expected by `spopt.locate`, we should pass a `numpy` 2D array as a cost matrix. So, first we pivot the network_distance dataframe. 

_Note that the columns and rows are sorted in ascending order._

In [ ]:
ntw_dist_piv = network_distance.pivot_table(
    values="distance", index="DestinationName", columns="name"
)
ntw_dist_piv

Here the pivot table is transformed to a numpy 2D array such as `spopt.locate` expected. The matrix has a shape of 205x16.

In [ ]:
cost_matrix = ntw_dist_piv.to_numpy()
cost_matrix.shape

In [ ]:
cost_matrix[:3, :3]

Now, as the rows and columns of our cost matrix are sorted, we have to sort our facility points and demand points geodataframes, too. 

In [ ]:
n_dem_pnts = demand_points.shape[0]
n_fac_pnts = facility_points.shape[0]

In [ ]:
process = lambda df: as_gdf(df).sort_values(by=["NAME"]).reset_index(drop=True)  # noqa: E731
as_gdf = lambda df: geopandas.GeoDataFrame(df, geometry=pnts(df))  # noqa: E731
pnts = lambda df: geopandas.points_from_xy(df.long, df.lat)  # noqa: E731

In [ ]:
facility_points = process(facility_points)
demand_points = process(demand_points)

Reproject the input spatial data.

In [ ]:
for _df in [facility_points, demand_points, study_area]:
    _df.set_crs("EPSG:4326", inplace=True)
    _df.to_crs("EPSG:7131", inplace=True)

Here the rest of parameters are set.

`ai` is the demand weight and in this case we model as population in year 2000 of this tract.

In [ ]:
ai = demand_points["POP2000"].to_numpy()

# maximum service radius (in meters)
SERVICE_RADIUS = 5000

# number of candidate facilities in optimal solution
P_FACILITIES = 4

Below, the method is used to plot the results of the four models that we will prepare to solve the problem.

In [ ]:
dv_colors_arr = [
    "darkcyan",
    "mediumseagreen",
    "saddlebrown",
    "darkslategray",
    "lightskyblue",
    "thistle",
    "lavender",
    "darkgoldenrod",
    "peachpuff",
    "coral",
    "mediumvioletred",
    "blueviolet",
    "fuchsia",
    "cyan",
    "limegreen",
    "mediumorchid",
]
dv_colors = {f"y{i}": dv_colors_arr[i] for i in range(len(dv_colors_arr))}
dv_colors

In [ ]:
def plot_results(model, p, facs, clis=None, ax=None):
    """Visualize optimal solution sets and context."""
    if not ax:
        multi_plot = False
        fig, ax = plt.subplots(figsize=(6, 9))
        markersize, markersize_factor = 4, 4
    else:
        ax.axis("off")
        multi_plot = True
        markersize, markersize_factor = 2, 2
    ax.set_title(model.name, fontsize=15)

    # extract facility-client relationships for plotting (except for p-dispersion)
    plot_clis = isinstance(clis, geopandas.GeoDataFrame)
    if plot_clis:
        cli_points = {}
    fac_sites = {}
    for i, dv in enumerate(model.fac_vars):
        if dv.varValue:
            dv, predef = facs.loc[i, ["dv", "predefined_loc"]]
            fac_sites[dv] = [i, predef]
            if plot_clis:
                geom = clis.iloc[model.fac2cli[i]]["geometry"]
                cli_points[dv] = geom

    # study area and legend entries initialization
    study_area.plot(ax=ax, alpha=0.5, fc="tan", ec="k", zorder=1)
    _patch = Patch(alpha=0.5, fc="tan", ec="k", label="Dissolved Service Areas")
    legend_elements = [_patch]

    if plot_clis and model.name.startswith("mclp"):
        # any clients that not asscociated with a facility
        c = "k"
        if model.n_cli_uncov:
            idx = [i for i, v in enumerate(model.cli2fac) if len(v) == 0]
            pnt_kws = {
                "ax": ax,
                "fc": c,
                "ec": c,
                "marker": "s",
                "markersize": 7,
                "zorder": 2,
            }
            clis.iloc[idx].plot(**pnt_kws)
        _label = f"Demand sites not covered ($n$={model.n_cli_uncov})"
        _mkws = {
            "marker": "s",
            "markerfacecolor": c,
            "markeredgecolor": c,
            "linewidth": 0,
        }
        legend_elements.append(mlines.Line2D([], [], ms=3, label=_label, **_mkws))

    # all candidate facilities
    facs.plot(ax=ax, fc="brown", marker="*", markersize=80, zorder=8)
    _label = f"Facility sites ($n$={len(model.fac_vars)})"
    _mkws = {"marker": "*", "markerfacecolor": "brown", "markeredgecolor": "brown"}
    legend_elements.append(mlines.Line2D([], [], ms=7, lw=0, label=_label, **_mkws))

    # facility-(client) symbology and legend entries
    zorder = 4
    for fname, (fac, predef) in fac_sites.items():
        cset = dv_colors[fname]
        if plot_clis:
            # clients
            geoms = cli_points[fname]
            gdf = geopandas.GeoDataFrame(geoms)
            gdf.plot(ax=ax, zorder=zorder, ec="k", fc=cset, markersize=100 * markersize)
            _label = f"Demand sites covered by {fname}"
            _mkws = {
                "markerfacecolor": cset,
                "markeredgecolor": "k",
                "ms": markersize + 7,
            }
            legend_elements.append(
                mlines.Line2D([], [], marker="o", lw=0, label=_label, **_mkws)
            )
        # facilities
        ec = "k"
        lw = 2
        predef_label = "predefined"
        if model.name.endswith(predef_label) and predef:
            ec = "r"
            lw = 3
            fname += f" ({predef_label})"
        facs.iloc[[fac]].plot(
            ax=ax, marker="*", markersize=1000, zorder=9, fc=cset, ec=ec, lw=lw
        )
        _mkws = {"markerfacecolor": cset, "markeredgecolor": ec, "markeredgewidth": lw}
        legend_elements.append(
            mlines.Line2D([], [], marker="*", ms=20, lw=0, label=fname, **_mkws)
        )
        # increment zorder up and markersize down for stacked client symbology
        zorder += 1
        if plot_clis:
            markersize -= markersize_factor / p

    if not multi_plot:
        # legend
        kws = {"loc": "upper left", "bbox_to_anchor": (1.05, 0.7)}
        plt.legend(handles=legend_elements, **kws)

-----------------------------------------

## LSCP

In [ ]:
from spopt.locate import LSCP

lscp = LSCP.from_cost_matrix(cost_matrix, SERVICE_RADIUS)
lscp = lscp.solve(pulp.GLPK(msg=False))
lscp_objval = lscp.problem.objective.value()
lscp_objval

Define the decision variable names used for mapping.

In [ ]:
facility_points["dv"] = lscp.fac_vars
facility_points["dv"] = facility_points["dv"].map(lambda x: x.name.replace("_", ""))
facility_points["predefined_loc"] = 0

In [ ]:
plot_results(lscp, lscp_objval, facility_points, clis=demand_points)

-----------------------------------------

## MCLP

In [ ]:
from spopt.locate import MCLP

mclp = MCLP.from_cost_matrix(
    cost_matrix,
    ai,
    service_radius=SERVICE_RADIUS,
    p_facilities=P_FACILITIES,
)
mclp = mclp.solve(pulp.GLPK(msg=False))
mclp.problem.objective.value()

MCLP Percentage Covered

In [ ]:
mclp.perc_cov

In [ ]:
plot_results(mclp, P_FACILITIES, facility_points, clis=demand_points)

-------------------------------------------------
## P-Median

In [ ]:
from spopt.locate import PMedian

pmedian = PMedian.from_cost_matrix(cost_matrix, ai, p_facilities=P_FACILITIES)
pmedian = pmedian.solve(pulp.GLPK(msg=False))
pmedian.problem.objective.value()

$p$-median mean weighted distance

In [ ]:
pmedian.mean_dist

In [ ]:
plot_results(pmedian, P_FACILITIES, facility_points, clis=demand_points)

--------------------------------


## P-Center

In [ ]:
from spopt.locate import PCenter

pcenter = PCenter.from_cost_matrix(cost_matrix, p_facilities=P_FACILITIES)
pcenter = pcenter.solve(pulp.GLPK(msg=False))
pcenter.problem.objective.value()

In [ ]:
plot_results(pcenter, P_FACILITIES, facility_points, clis=demand_points)

-----------------------------------

## Comparing all models

In [ ]:
fig, axarr = plt.subplots(1, 4, figsize=(20, 10))
fig.subplots_adjust(wspace=-0.01)
for i, m in enumerate([lscp, mclp, pmedian, pcenter]):
    _p = m.problem.objective.value() if m.name == "lscp" else P_FACILITIES
    plot_results(m, _p, facility_points, clis=demand_points, ax=axarr[i])

The plot above clearly demonstrates the key features of each location model:
* LSCP
  * Overlapping clusters of clients associated with multiple facilities
  * All clients covered by at least one facility
* MCLP
  * Overlapping clusters of clients associated with multiple facilities
  * Potential for complete coverage, but not guranteed
* $p$-median
  * Tight and distinct clusters of client-facility relationships
  * All clients allocated to exactly one facility
* $p$-center
  * Loose and overlapping clusters of client-facility relationships
  * All clients allocated to exactly one facility 

----------------------------------------

## Evaluating available solvers

For this task we'll solve the MCLP and $p$-center models as examples of solver performance, but first we'll determine which solvers are installed locally.

In [ ]:
with warnings.catch_warnings(record=True) as w:
    solvers = pulp.listSolvers(onlyAvailable=True)
for _w in w:
    print(_w.message)
solvers

Above we can see that it returns a list with different solvers that are available. So, it's up to the user to choose the best solver that fits the model. 

### MCLP

Let's get the percentage as a metric to evaluate which solver is the best or improves the model.

In [ ]:
mclp = MCLP.from_cost_matrix(
    cost_matrix,
    ai,
    service_radius=SERVICE_RADIUS,
    p_facilities=P_FACILITIES,
)

In [ ]:
results = pandas.DataFrame(columns=["Coverage %", "Solve Time (sec.)"], index=solvers)
for solver in solvers:
    _mclp = mclp.solve(pulp.getSolver(solver, msg=False))
    results.loc[solver] = [_mclp.perc_cov, _mclp.problem.solutionTime]
results

As is expected from such a small model the **Coverage %**, and thus the objective values, are equivalent for all solutions across solvers. However, solve times do vary slightly.

### P-Center

In [ ]:
pcenter = PCenter.from_cost_matrix(cost_matrix, p_facilities=P_FACILITIES)

In [ ]:
results = pandas.DataFrame(columns=["MinMax", "Solve Time (sec.)"], index=solvers)
for solver in solvers:
    _pcenter = pcenter.solve(pulp.getSolver(solver, msg=False))
    results.loc[solver] = [
        _pcenter.problem.objective.value(),
        _pcenter.problem.solutionTime,
    ]
results

Here the **MinMax** objective values are mostly equivalent, but there are some precision concerns. This is due to the computational complexity of the $p$-center problem, which also results in dramaticaly longer run times needed to isolate the optimal solutions (when compared with the MCLP above).